## 目標：地震の発生場所をマップにする

In [11]:
import pandas as pd
import requests

# USGSから過去1か月の地震データを取得
url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_month.csv"
response = requests.get(url)

# データをファイルに保存
with open('earthquakes.csv', 'wb') as file:
    file.write(response.content)

# データをDataFrameに読み込む
data = pd.read_csv('earthquakes.csv')

# データの最初の5行を表示
data.head()

,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2025-12-23T05:22:39.120Z,17.930167,-66.912000,11.35,1.810000,md,5.0,254.0,0.05445,0.06,...,2025-12-23T05:42:29.640Z,"4 km S of Guánica, Puerto Rico",earthquake,1.33,0.64,0.101596,4.0,reviewed,pr,pr
1,2025-12-23T05:15:52.710Z,33.044834,-116.372498,9.98,1.484259,ml,11.0,127.0,0.07956,0.07,...,2025-12-23T05:17:57.510Z,"22 km E of Julian, CA",earthquake,0.50,1.20,0.323743,11.0,automatic,ci,ci
2,2025-12-23T05:14:29.700Z,35.338500,-117.797667,5.36,1.000000,ml,33.0,81.0,0.08112,0.17,...,2025-12-23T05:18:09.393Z,"15 km WSW of Johannesburg, CA",earthquake,0.20,0.67,0.137185,13.0,automatic,ci,ci
3,2025-12-23T05:04:04.070Z,35.338667,-117.795000,5.90,1.170000,ml,35.0,81.0,0.08234,0.17,...,2025-12-23T05:07:43.042Z,"15 km WSW of Johannesburg, CA",earthquake,0.19,0.60,0.169144,21.0,automatic,ci,ci
4,2025-12-23T04:54:44.110Z,33.764500,-117.540500,7.73,0.670000,ml,14.0,86.0,0.06484,0.10,...,2025-12-23T04:58:24.161Z,"12 km SSE of Corona, CA",earthquake,0.24,0.60,0.274213,11.0,automatic,ci,ci


緯度経度とマグニチュードのデータからマップを作ります。

In [12]:
import folium
from folium.plugins import HeatMap
import pandas as pd

# データの読み込み
df = pd.read_csv('earthquakes.csv')

# 欠損値（緯度・経度がないデータ）を削除
df = df.dropna(subset=['latitude', 'longitude'])

# 地図の初期位置（データの平均位置）
m = folium.Map(location=[df['latitude'].mean(), df['longitude'].mean()], 
               zoom_start=2, 
               tiles='CartoDB positron') # 見やすい薄い色の地図

# --- オプション A: 震源地にサークルマーカーを配置 ---
for i, row in df.head(2000).iterrows():  # 負荷軽減のため2000件
    # マグニチュードによって色を変える
    color = 'red' if row['mag'] >= 5 else 'orange' if row['mag'] >= 3 else 'blue'
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=row['mag'] * 1.5, # マグニチュードに応じてサイズ変更
        popup=f"Mag: {row['mag']}<br>Depth: {row['depth']}km",
        color=color,
        fill=True,
        fill_opacity=0.6
    ).add_to(m)

# 地図をHTMLとして保存
m.save('earthquake_map.html')

## 分析結果のマップ
![地震マップの表示](earthquake_map.png)

せっかくだからマグニチュードによって色を変えてみました。
5以上が赤、5未満3以上がオレンジ、3未満が青です。